# Task 2.3 (Final Fixed Version)
Run all cells step by step

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)


## Load your data file

In [ ]:
load_profiles = pd.read_csv('load_profiles_step2.csv')
load_profiles.head()

## Compute availability

In [ ]:
availability = (load_profiles.groupby(['profile','set'])['load_kW']
 .min()
 .reset_index()
 .rename(columns={'load_kW':'availability_kW'}))
availability.head()

## Split data

In [ ]:
in_sample = availability[availability['set']=='in_sample']['availability_kW'].values
out_sample = availability[availability['set']=='out_of_sample']['availability_kW'].values


## Correct ALSO-X function

In [ ]:
def alsox(avail, rel):
    avail_sorted = np.sort(avail)
    n = len(avail_sorted)

    if rel >= 0.999:
        return avail_sorted[0]

    k = int(np.floor((1 - rel) * n))
    k = min(max(k, 0), n - 1)

    return avail_sorted[k]


## Run Task 2.3

In [ ]:
rows = []
for r in np.arange(0.80,1.01,0.01):
    bid = alsox(in_sample, r)
    shortfall = np.maximum(0, bid - out_sample)
    rows.append([r, bid, np.mean(shortfall)])

res = pd.DataFrame(rows, columns=['Reliability','Bid','Expected Shortfall'])
res


## Plot results

In [ ]:
plt.figure()
plt.plot(res['Reliability'], res['Bid'], marker='o')
plt.xlabel('Reliability')
plt.ylabel('Reserve Bid (kW)')
plt.title('Reliability vs Reserve Bid')
plt.grid()
plt.show()

plt.figure()
plt.plot(res['Reliability'], res['Expected Shortfall'], marker='o')
plt.xlabel('Reliability')
plt.ylabel('Expected Shortfall (kW)')
plt.title('Reliability vs Shortfall')
plt.grid()
plt.show()


## Save results

In [ ]:
res.to_csv('task_2_3_results.csv', index=False)
print('Saved as task_2_3_results.csv')